In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from pprint import pprint
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
%matplotlib inline

In [3]:
cwd = Path.cwd()
project_root = cwd.parent
data_path = project_root / "data" / "Online Retail.xlsx"

In [4]:
data = pd.ExcelFile(str(data_path))
data.sheet_names

['Online Retail']

In [5]:
data = data.parse("Online Retail")

In [6]:
data = data[data["Quantity"] > 0]

In [7]:
latest_date = max(data["InvoiceDate"])
latest_date

Timestamp('2011-12-09 12:50:00')

In [8]:
threshold_time = latest_date - pd.Timedelta(days=30)

In [9]:
t_data = data[data["InvoiceDate"] < threshold_time]

In [10]:
t_data.info()

<class 'pandas.DataFrame'>
Index: 442614 entries, 0 to 452024
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    442614 non-null  object        
 1   StockCode    442614 non-null  object        
 2   Description  442081 non-null  object        
 3   Quantity     442614 non-null  int64         
 4   InvoiceDate  442614 non-null  datetime64[us]
 5   UnitPrice    442614 non-null  float64       
 6   CustomerID   332343 non-null  float64       
 7   Country      442614 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 30.4+ MB


In [11]:
t_data.isna().all()

InvoiceNo      False
StockCode      False
Description    False
Quantity       False
InvoiceDate    False
UnitPrice      False
CustomerID     False
Country        False
dtype: bool

In [16]:
cust_new_data = pd.DataFrame()

In [17]:
cust_new_data["NoOfUniqueProds"] = t_data.groupby("CustomerID")["StockCode"].nunique()

In [19]:
cust_new_data.head()

,NoOfUniqueProds
CustomerID,
12346.0,1
12347.0,100
12348.0,22
12350.0,17
12352.0,59


In [20]:
t_data["TotalOrderValue"] = t_data["Quantity"].multiply(t_data["UnitPrice"])

In [23]:
order_totals = t_data.groupby(["CustomerID", "InvoiceNo"])["TotalOrderValue"].sum()

In [24]:
order_totals

CustomerID  InvoiceNo
12346.0     541431       77183.60
12347.0     537626         711.79
            542237         475.39
            549222         636.25
            556201         382.52
                           ...   
18283.0     565579         134.90
            573093         114.65
18287.0     554065         765.28
            570715        1001.32
            573167          70.68
Name: TotalOrderValue, Length: 15797, dtype: float64

In [25]:
cust_new_data["AverageOrderValue"] = order_totals.groupby("CustomerID").mean()

In [26]:
cust_new_data

,NoOfUniqueProds,AverageOrderValue
CustomerID,,
12346.0,1,77183.600000
12347.0,100,680.863333
12348.0,22,449.310000
12350.0,17,334.400000
12352.0,59,313.255000
...,...,...
18280.0,10,180.600000
18281.0,7,80.820000
18282.0,7,100.210000


In [39]:
def get_time_of_day(hour):
    if 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 18:
        return "Afternoon"
    elif 18 <= hour < 24:
        return "Evening"
    else:
        return "Night"

In [43]:
t_data["TimeofDay"] = t_data["InvoiceDate"].apply(lambda x: get_time_of_day(x.hour))

In [49]:
cust_new_data["FavTime"] = t_data.groupby("CustomerID")["TimeofDay"].agg(lambda x: x.mode().iloc[0])

In [50]:
cust_new_data

,NoOfUniqueProds,AverageOrderValue,FavTime
CustomerID,,,
12346.0,1,77183.600000,Morning
12347.0,100,680.863333,Afternoon
12348.0,22,449.310000,Evening
12350.0,17,334.400000,Afternoon
12352.0,59,313.255000,Afternoon
...,...,...,...
18280.0,10,180.600000,Morning
18281.0,7,80.820000,Morning
18282.0,7,100.210000,Afternoon


In [53]:
pd.get_dummies(cust_new_data["FavTime"])

,Afternoon,Evening,Morning
CustomerID,,,
12346.0,False,False,True
12347.0,True,False,False
12348.0,False,True,False
12350.0,True,False,False
12352.0,True,False,False
...,...,...,...
18280.0,False,False,True
18281.0,False,False,True
18282.0,True,False,False


In [56]:
cust_new_data[["Afternoon", "Evening", "Morning"]] = pd.get_dummies(cust_new_data["FavTime"], dtype=int)

In [58]:
cust_new_data.head()

,NoOfUniqueProds,AverageOrderValue,FavTime,Afternoon,Evening,Morning
CustomerID,,,,,,
12346.0,1,77183.600000,Morning,0,0,1
12347.0,100,680.863333,Afternoon,1,0,0
12348.0,22,449.310000,Evening,0,1,0
12350.0,17,334.400000,Afternoon,1,0,0
12352.0,59,313.255000,Afternoon,1,0,0


In [59]:
cust_new_data.drop(columns=["FavTime"], inplace=True)

In [67]:
t_data["Day"] = t_data["InvoiceDate"].dt.day_name()

In [68]:
cust_new_data["DayOfWeek"] = t_data.groupby("CustomerID")["Day"].agg(lambda x: x.mode().iloc[0])

In [70]:
cust_new_data

,NoOfUniqueProds,AverageOrderValue,Afternoon,Evening,Morning,DayOfWeek
CustomerID,,,,,,
12346.0,1,77183.600000,0,0,1,Tuesday
12347.0,100,680.863333,1,0,0,Tuesday
12348.0,22,449.310000,0,1,0,Thursday
12350.0,17,334.400000,1,0,0,Wednesday
12352.0,59,313.255000,1,0,0,Tuesday
...,...,...,...,...,...,...
18280.0,10,180.600000,0,0,1,Monday
18281.0,7,80.820000,0,0,1,Sunday
18282.0,7,100.210000,1,0,0,Friday


In [73]:
cust_new_data[["Friday", "Monday", "Sunday", "Thursday", "Tuesday", "Wednesday"]] = pd.get_dummies(cust_new_data["DayOfWeek"], dtype=int)

In [74]:
cust_new_data.drop(columns=["DayOfWeek"], inplace=True)

In [75]:
cust_new_data.head()

,NoOfUniqueProds,AverageOrderValue,Afternoon,Evening,Morning,Friday,Monday,Sunday,Thursday,Tuesday,Wednesday
CustomerID,,,,,,,,,,,
12346.0,1,77183.600000,0,0,1,0,0,0,0,1,0
12347.0,100,680.863333,1,0,0,0,0,0,0,1,0
12348.0,22,449.310000,0,1,0,0,0,0,1,0,0
12350.0,17,334.400000,1,0,0,0,0,0,0,0,1
12352.0,59,313.255000,1,0,0,0,0,0,0,1,0


In [77]:
cust_new_data.to_csv(project_root / "data" / "new_features.csv")